In [22]:
import geopandas as gpd
import pandas as pd
import re

# Load data
retail_landuse_h3 = gpd.read_parquet("../data/input_files/retail_landuse_h3_geo.parquet")
retail_landuse = gpd.read_parquet("../data/input_files/retail_landuse.parquet")
foursquare_pois = gpd.read_parquet("../data/input_files/uk_foursquare_cc_aug_retailfilt_geo.parquet")
retail_park_stores = pd.read_csv("../data/input_files/bl_retail_parks_brands_deduped.csv")

# --- Regex-based whole-word filtering ---

# Helper: turn a brand into a whole-word regex (case-insensitive, flexible whitespace)
def brand_to_pattern(brand: str) -> str:
    brand = str(brand).strip().lower()
    tokens = re.split(r"\s+", brand)
    return r"\b" + r"\s+".join(re.escape(t) for t in tokens if t) + r"\b"

# Build a single alternation regex for all brands
brands = retail_park_stores["brand"].dropna().astype(str).str.strip().str.lower().unique()
brand_patterns = [brand_to_pattern(b) for b in brands if b]
big_pattern = r"(?i)(" + "|".join(brand_patterns) + r")"  # (?i) = case-insensitive

# Apply to Foursquare names
name_series = foursquare_pois["name"].astype(str)
mask = name_series.str.contains(big_pattern, regex=True, na=False)

# Filtered POIs: only those where a full brand word/phrase occurs
foursquare_pois_filtered = foursquare_pois[mask].copy()
foursquare_pois_filtered["matched_brand_text"] = name_series[mask].str.extract(big_pattern, expand=False)

# --- Count matches per brand ---

# Normalize matched text for grouping
foursquare_pois_filtered["matched_brand_norm"] = (
    foursquare_pois_filtered["matched_brand_text"].str.lower().str.strip()
)

# Count how many times each brand occurs
brand_counts = (
    foursquare_pois_filtered["matched_brand_norm"]
    .value_counts()
    .rename_axis("brand_norm")
    .reset_index(name="match_count")
)

# Merge back into retail_park_stores
retail_park_stores["brand_norm"] = retail_park_stores["brand"].str.lower().str.strip()
retail_park_stores_with_counts = retail_park_stores.merge(
    brand_counts, on="brand_norm", how="left"
)

# Fill NaN with 0 (no matches found) and sort
retail_park_stores_with_counts["match_count"] = (
    retail_park_stores_with_counts["match_count"].fillna(0).astype(int)
)
retail_park_stores_with_counts_sorted = (
    retail_park_stores_with_counts
    .sort_values("match_count", ascending=False)
    .reset_index(drop=True)
)


/tmp/ipykernel_145471/1972614120.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = name_series.str.contains(big_pattern, regex=True, na=False)


In [23]:
#looks decent but "Mango" getting some extras (this is a clothing outlet) 
#I could mannually get the cat(s) for each of the brands and further filter with that
foursquare_pois_filtered

,fsq_place_id,name,lat,lng,address,locality,region,postcode,admin_region,post_town,...,date_refreshed,date_closed,tel,website,fsq_category_id,fsq_category_labels,h3_11,geometry,matched_brand_text,matched_brand_norm
46,c8b48b566b404f3a74b4bf5f,Blue Mango,50.548282,-4.146221,19 Plymouth Rd,Tavistock,Devon,PL19 8AU,England,Tavistock,...,2025-06-05,None,01822 617581,http://www.bluemango.org.uk,52e81612bcbc57f1066b7a05,[Dining and Drinking > Restaurant > English Re...,8b1874a10a25fff,POINT (-4.14622 50.54828),Mango,mango
73,5d2d94c0cdfaa90023bc7804,Argos,54.564404,-5.910790,Forestside Shopping Centre,Antrim,Antrim,BT8 6FX,Northern Ireland,None,...,2023-07-02,None,0345 266 1487,https://www.argos.co.uk,52e816a6bcbc57f1066b7a54,[Retail > Warehouse or Wholesale Store],8b1952d6b0acfff,POINT (-5.91079 54.5644),Argos,argos
83,c28d75e2458046ef777a2b05,Costa Express,52.993600,-3.031170,Ruabon Bypass,Wrexham,Wrexham,None,None,Wrexham,...,2023-04-23,None,None,https://www.costa.co.uk,4bf58dd8d48988d1e0931735,"[Dining and Drinking > Cafe, Coffee, and Tea H...",8b19512d02d8fff,POINT (-3.03117 52.9936),Costa,costa
94,4c8a3cff1eafb1f74e4a7835,Furniture Village,51.483930,0.273460,"The Junction, Thurrock Retail Park",Thurrock,Thurrock,RM20 3LP,England,None,...,2025-01-03,None,01708 899421,https://www.furniturevillage.co.uk,4bf58dd8d48988d1f8941735,[Retail > Furniture and Home Store],8b194e6ecd25fff,POINT (0.27346 51.48393),Furniture Village,furniture village
129,4d8ba4fbf29a43598cd68404,Greggs,54.618119,-1.066352,U3/4 Regents Walk Unit 4,Redcar,North Yorkshire,TS10 3FB,England,None,...,2025-07-04,None,01642 493692,https://www.greggs.co.uk,4bf58dd8d48988d16a941735,[Dining and Drinking > Bakery],8b1946191ae0fff,POINT (-1.06635 54.61812),Greggs,greggs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1492692,4c0a5c0ea1b32d7f362099f0,McDonald's,53.660329,-1.762626,501 Leeds Rd,Huddersfield,West Yorkshire,HD2 1YJ,England,None,...,2025-07-04,None,01484 451406,https://www.mcdonalds.com,4bf58dd8d48988d16e941735,[Dining and Drinking > Restaurant > Fast Food ...,8b194256b6a5fff,POINT (-1.76263 53.66033),McDonald's,mcdonald's
1492703,17019c38e660421329675e9e,Boots,51.892880,-1.163790,Bicester Shopping Park Unit A,Bicester,Oxfordshire,OX26 1ES,England,Bicester,...,2025-04-05,None,01869 357812,https://www.boots.com,4bf58dd8d48988d10f951735,[Retail > Pharmacy],8b195d1985b2fff,POINT (-1.16379 51.89288),Boots,boots
1492713,5656f0dd498e9760a5afebf6,Sainsbury's Local,57.226406,-2.345844,Midmill Kintore,Kintore,Aberdeenshire,AB51 0UY,Scotland,None,...,2025-06-07,None,01467 632537,https://stores.sainsburys.co.uk/4566/kintore-m...,4bf58dd8d48988d118951735,[Retail > Food and Beverage Retail > Grocery S...,8b1976125208fff,POINT (-2.34584 57.22641),Sainsbury's,sainsbury's
1492740,8a31f29fa78241a58a5c28e2,Costa Express,51.611872,0.537237,155 South End Rd,Wickford,Essex,None,England,Wickford,...,2023-04-23,None,None,https://www.costa.co.uk,4bf58dd8d48988d1e0931735,"[Dining and Drinking > Cafe, Coffee, and Tea H...",8b194e76e4cdfff,POINT (0.53724 51.61187),Costa,costa


In [ ]:
#

,fsq_place_id,name,lat,lng,address,locality,region,postcode,admin_region,post_town,...,date_refreshed,date_closed,tel,website,fsq_category_id,fsq_category_labels,h3_11,geometry,name_norm,matched_brand_text
46,c8b48b566b404f3a74b4bf5f,Blue Mango,50.548282,-4.146221,19 Plymouth Rd,Tavistock,Devon,PL19 8AU,England,Tavistock,...,2025-06-05,None,01822 617581,http://www.bluemango.org.uk,52e81612bcbc57f1066b7a05,[Dining and Drinking > Restaurant > English Re...,8b1874a10a25fff,POINT (-4.14622 50.54828),blue mango,Mango
73,5d2d94c0cdfaa90023bc7804,Argos,54.564404,-5.910790,Forestside Shopping Centre,Antrim,Antrim,BT8 6FX,Northern Ireland,None,...,2023-07-02,None,0345 266 1487,https://www.argos.co.uk,52e816a6bcbc57f1066b7a54,[Retail > Warehouse or Wholesale Store],8b1952d6b0acfff,POINT (-5.91079 54.5644),argos,Argos
83,c28d75e2458046ef777a2b05,Costa Express,52.993600,-3.031170,Ruabon Bypass,Wrexham,Wrexham,None,None,Wrexham,...,2023-04-23,None,None,https://www.costa.co.uk,4bf58dd8d48988d1e0931735,"[Dining and Drinking > Cafe, Coffee, and Tea H...",8b19512d02d8fff,POINT (-3.03117 52.9936),costa express,Costa
94,4c8a3cff1eafb1f74e4a7835,Furniture Village,51.483930,0.273460,"The Junction, Thurrock Retail Park",Thurrock,Thurrock,RM20 3LP,England,None,...,2025-01-03,None,01708 899421,https://www.furniturevillage.co.uk,4bf58dd8d48988d1f8941735,[Retail > Furniture and Home Store],8b194e6ecd25fff,POINT (0.27346 51.48393),furniture village,Furniture Village
129,4d8ba4fbf29a43598cd68404,Greggs,54.618119,-1.066352,U3/4 Regents Walk Unit 4,Redcar,North Yorkshire,TS10 3FB,England,None,...,2025-07-04,None,01642 493692,https://www.greggs.co.uk,4bf58dd8d48988d16a941735,[Dining and Drinking > Bakery],8b1946191ae0fff,POINT (-1.06635 54.61812),greggs,Greggs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1492692,4c0a5c0ea1b32d7f362099f0,McDonald's,53.660329,-1.762626,501 Leeds Rd,Huddersfield,West Yorkshire,HD2 1YJ,England,None,...,2025-07-04,None,01484 451406,https://www.mcdonalds.com,4bf58dd8d48988d16e941735,[Dining and Drinking > Restaurant > Fast Food ...,8b194256b6a5fff,POINT (-1.76263 53.66033),mcdonald's,McDonald's
1492703,17019c38e660421329675e9e,Boots,51.892880,-1.163790,Bicester Shopping Park Unit A,Bicester,Oxfordshire,OX26 1ES,England,Bicester,...,2025-04-05,None,01869 357812,https://www.boots.com,4bf58dd8d48988d10f951735,[Retail > Pharmacy],8b195d1985b2fff,POINT (-1.16379 51.89288),boots,Boots
1492713,5656f0dd498e9760a5afebf6,Sainsbury's Local,57.226406,-2.345844,Midmill Kintore,Kintore,Aberdeenshire,AB51 0UY,Scotland,None,...,2025-06-07,None,01467 632537,https://stores.sainsburys.co.uk/4566/kintore-m...,4bf58dd8d48988d118951735,[Retail > Food and Beverage Retail > Grocery S...,8b1976125208fff,POINT (-2.34584 57.22641),sainsbury's local,Sainsbury's
1492740,8a31f29fa78241a58a5c28e2,Costa Express,51.611872,0.537237,155 South End Rd,Wickford,Essex,None,England,Wickford,...,2023-04-23,None,None,https://www.costa.co.uk,4bf58dd8d48988d1e0931735,"[Dining and Drinking > Cafe, Coffee, and Tea H...",8b194e76e4cdfff,POINT (0.53724 51.61187),costa express,Costa
